# Classical Fake-News Classification in Google Colab

This notebook trains **only Logistic Regression and calibrated Linear SVM** using the repository's prepared datasets and reproducible TF-IDF pipeline.

The notebook treats model probabilities as content-classification signals—not proof that a story is true or false. Models, metrics, and other generated files are stored in Google Drive so they survive Colab runtime resets.

## Before running

1. Place the existing project `data` folder in `MyDrive/hybrid-fake-news-detection/data`. It must contain `raw`, `interim`, and `processed`, including `processed/manifest.json`.
2. Run the notebook from top to bottom. A CPU runtime is sufficient; GPU is not required.

In [ ]:
from pathlib import Path

REPO_OWNER = "hardiktyagi032"
REPO_NAME = "hybrid-fake-news-detection"
REPO_DIR = Path("/content") / REPO_NAME
DRIVE_PROJECT_DIR = Path("/content/drive/MyDrive/hybrid-fake-news-detection")
DRIVE_DATA_DIR = DRIVE_PROJECT_DIR / "data"
DRIVE_MODEL_DIR = DRIVE_PROJECT_DIR / "models"
APPROVED_MODELS = ("logistic_regression", "linear_svm")
TRAINING_DATASETS = ("isot", "ifnd", "indian_fake_news", "bharatgather")
MODEL_SET_NAME = "multidataset_v2_robust"

print(f"Repository workspace: {REPO_DIR}")
print(f"Persistent data: {DRIVE_DATA_DIR}")
print(f"Persistent models: {DRIVE_MODEL_DIR}")

## 1. Mount Google Drive

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
DRIVE_MODEL_DIR.mkdir(parents=True, exist_ok=True)

required_data_paths = [
    DRIVE_DATA_DIR / "raw",
    DRIVE_DATA_DIR / "interim",
    DRIVE_DATA_DIR / "processed",
    DRIVE_DATA_DIR / "processed" / "manifest.json",
]
missing = [str(path) for path in required_data_paths if not path.exists()]
if missing:
    raise FileNotFoundError(
        "Project data is missing from Google Drive:\n" + "\n".join(missing)
    )
print("Google Drive is mounted and the project data layout is present.")

## 2. Clone the public GitHub repository

In [ ]:
import os
import shutil
import subprocess

os.chdir("/content")
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

clone_url = f"https://github.com/{REPO_OWNER}/{REPO_NAME}.git"
clone = subprocess.run(
    ["git", "clone", "--depth", "1", clone_url, str(REPO_DIR)],
    capture_output=True,
    text=True,
)
if clone.returncode != 0:
    raise RuntimeError(f"Repository clone failed:\n{clone.stderr}")
print("Public repository cloned successfully.")

## 3. Connect persistent data and model storage

The fresh clone contains only placeholder data directories. This cell links its `data` and `models` paths to the corresponding Google Drive folders.

In [ ]:
import os

def replace_with_directory_link(target: Path, source: Path) -> None:
    if target.is_symlink() or target.is_file():
        target.unlink()
    elif target.is_dir():
        shutil.rmtree(target)
    os.symlink(source, target, target_is_directory=True)

replace_with_directory_link(REPO_DIR / "data", DRIVE_DATA_DIR)
replace_with_directory_link(REPO_DIR / "models", DRIVE_MODEL_DIR)
print("Repository data and model paths now use persistent Drive storage.")

## 4. Install the classical ML environment

In [ ]:
import sys

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_DIR / "requirements-dev.txt")],
    check=True,
)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_DIR / "requirements-ml.txt")],
    check=True,
)
print("Dependencies installed.")

## 5. Verify code and prepared data

Training stops if the tests fail, the prepared outputs changed, or the original raw archive hashes no longer match the Phase 4 manifest.

In [ ]:
import os

os.chdir(REPO_DIR)
subprocess.run([sys.executable, "-m", "pytest", "-q"], check=True)
subprocess.run([sys.executable, "scripts/verify_setup.py"], check=True)
subprocess.run(
    [sys.executable, "scripts/validate_prepared_data.py", "--root", str(REPO_DIR)],
    check=True,
)
print("Code tests and dataset integrity checks passed.")

## 6. Train source-robust Logistic Regression and Linear SVM v2

The v2 workflow masks known publisher boilerplate and adds only LIAR's clear endpoint labels at one-quarter dataset weight. Model and threshold selection use dataset-held-out validation folds. This is intentionally more rigorous and slower than a random within-dataset split.

In [ ]:
train_command = [
    sys.executable,
    "-m",
    "scripts.train_classical_robust",
    "--root",
    str(REPO_DIR),
    "--datasets",
    *TRAINING_DATASETS,
    "--models",
    *APPROVED_MODELS,
]
subprocess.run(train_command, check=True)

## 7. Display source-disjoint validation selection

In [ ]:
import json
import pandas as pd
from IPython.display import display

benchmark_path = DRIVE_MODEL_DIR / "classical" / MODEL_SET_NAME / "benchmark.json"
benchmark = json.loads(benchmark_path.read_text(encoding="utf-8"))
rows = []
for model_name, validation_result in benchmark["validation"].items():
    metrics = validation_result["selected_threshold"]
    rows.append({
        "model": model_name,
        "threshold": metrics["threshold"],
        "mean_dataset_macro_f1": metrics["mean_dataset_macro_f1"],
        "worst_dataset_macro_f1": metrics["worst_dataset_macro_f1"],
    })
display(pd.DataFrame(rows).round(4))
print(f"Preferred source-robust model: {benchmark['selected_model']}")
print(f"Full report: {benchmark_path}")

## 8. Display held-out results for every dataset

These test splits were not used for v2 model or threshold selection. They were inspected during earlier baseline work, so they are not pristine for future iteration. Compare balanced accuracy and macro-F1 because the class distributions differ.

In [ ]:
test_rows = []
for model_name, model_result in benchmark["held_out_test_metrics"].items():
    for dataset_name, metrics in model_result.items():
        test_rows.append({
            "dataset": dataset_name,
            "model": model_name,
            "records": metrics["records"],
            "accuracy": metrics["accuracy"],
            "balanced_accuracy": metrics["balanced_accuracy"],
            "macro_f1": metrics["f1_macro"],
            "roc_auc": metrics["roc_auc"],
            "brier_score": metrics["brier_score"],
        })

In [ ]:
display(pd.DataFrame(test_rows).sort_values(["dataset", "model"]).round(4))
for model_name, metrics in benchmark["held_out_test_metrics"].items():
    macro_f1_values = [item["f1_macro"] for item in metrics.values()]
    print(
        model_name,
        "mean dataset macro-F1:",
        round(sum(macro_f1_values) / len(macro_f1_values), 4),
        "worst dataset macro-F1:",
        round(min(macro_f1_values), 4),
    )

## 9. Optional inference demonstration

Edit the title and article below. The outputs use each model's validation-selected threshold and remain content-pattern signals—not verified factuality judgments.

In [ ]:
example_title = "Official agency releases new report"
example_article = (
    "The agency published an audited report with named sources and links to the underlying records."
)
prediction_rows = []
for model_name in APPROVED_MODELS:
    model_path = DRIVE_MODEL_DIR / "classical" / MODEL_SET_NAME / model_name / "pipeline.joblib"
    threshold = benchmark["validation"][model_name]["selected_threshold"]["threshold"]
    prediction = subprocess.run(
        [
            sys.executable,
            "-m",
            "scripts.predict_classical",
            "--model-path",
            str(model_path),
            "--title",
            example_title,
            "--text",
            example_article,
            "--threshold",
            str(threshold),
        ],
        check=True,
        capture_output=True,
        text=True,
    )
    prediction_rows.append(json.loads(prediction.stdout))
display(pd.DataFrame(prediction_rows).round(4))
print("Warning: this is content classification, not evidence-based fact verification.")

## 10. Evaluate the frozen contemporary diagnostic

This 20-claim set was created after model and threshold selection. It is small and hand-curated, so treat it as a diagnostic rather than a benchmark. It is never used for training.

In [ ]:
subprocess.run(
    [sys.executable, "-m", "scripts.evaluate_contemporary", "--root", str(REPO_DIR)],
    check=True,
)
contemporary_path = DRIVE_MODEL_DIR / "classical" / "contemporary_2026_evaluation.json"
contemporary = json.loads(contemporary_path.read_text(encoding="utf-8"))
display(pd.DataFrame.from_dict(contemporary["results"], orient="index").round(4))
print("Diagnostic limitations:")
for limitation in contemporary["limitations"]:
    print("-", limitation)

## Outputs

Persistent outputs are stored under `MyDrive/hybrid-fake-news-detection/models/classical/multidataset_v2_robust/`:

- `logistic_regression/pipeline.joblib`
- `linear_svm/pipeline.joblib`
- `benchmark.json` with source-disjoint validation, selected thresholds, and per-dataset test metrics

Do not commit model binaries or datasets to Git.